# Level 3 — EDA dan representasi teks (Tahap C proyek)

> **Data sintetis.** Dataset dibangkitkan dari templat; set tulis-tangan ditulis penulis materi
> (lihat `../disaster-classification/DATA_CARD.md`). Temuan di sini **tidak** mewakili data Twitter/X nyata.

Prasyarat: `pip install -r requirements-ml.txt`. Jalankan sel dari atas ke bawah.

In [1]:
import sys
import warnings
from pathlib import Path

# Notebook ini berada di examples/<folder-level>/. Paket nlp_hub ada di examples/.
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", message="'count' is passed as positional argument")

## 1. Struktur, nilai kosong, duplikat, label

In [2]:
from nlp_hub.data import muat_dataset

mentah = muat_dataset(hapus_duplikat=False)
print(mentah.shape)
print("nilai kosong:", int(mentah.isna().sum().sum()), "| duplikat teks:", int(mentah["text"].duplicated().sum()))
print(mentah["label"].value_counts(normalize=True).round(3).to_dict())
df = muat_dataset()  # duplikat persis dibuang
print("setelah dedup:", df.shape)

(574, 6)
nilai kosong: 0 | duplikat teks: 10
{'Non-Emergency': 0.65, 'Emergency': 0.35}
setelah dedup: (564, 6)


## 2. Panjang teks per kelas
Perbedaan panjang berasal dari templat (laporan memuat alamat); jangan digeneralisasi.

In [3]:
df["n_karakter"] = df["text"].str.len()
df.groupby("label")["n_karakter"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
Emergency,191.0,96.6,16.4,63.0,85.0,95.0,104.0,148.0
Non-Emergency,373.0,81.1,20.8,40.0,66.0,79.0,93.0,145.0


## 3. Kata paling sering (document frequency) per kelas

In [4]:
from collections import Counter
from nlp_hub.preprocessing import praproses_token

token = [set(praproses_token(t)) for t in df["text"]]
for label in ["Emergency", "Non-Emergency"]:
    c = Counter(k for s, l in zip(token, df["label"]) if l == label for k in s)
    print(label, c.most_common(8))

Emergency [('di', 148), ('rt', 120), ('jalan', 114), ('dan', 77), ('kelurahan', 71), ('rumah', 65), ('warga', 46), ('butuh', 43)]
Non-Emergency [('di', 103), ('kelurahan', 77), ('ini', 68), ('tok_mention', 63), ('sudah', 58), ('bencana', 57), ('gunung', 53), ('tok_url', 52)]


## 4. Bag of Words dan TF-IDF
Vektorizer **hanya** dilatih pada data latih (`fit`), lalu dipakai pada data uji (`transform`).
Pembagian dilakukan per `event_id` agar satu kejadian tidak terpecah.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nlp_hub.data import bagi_latih_uji
from nlp_hub.preprocessing import PraprosesTeks

latih, uji = bagi_latih_uji(df)
v = TfidfVectorizer(min_df=2)
X_latih = v.fit_transform(PraprosesTeks().transform(latih["text"]))
X_uji = v.transform(PraprosesTeks().transform(uji["text"]))
print("latih:", X_latih.shape, "uji:", X_uji.shape, f"kepadatan latih = {X_latih.nnz / (X_latih.shape[0] * X_latih.shape[1]):.2%}")
print("kejadian di kedua split:", len(set(latih["event_id"]) & set(uji["event_id"])))

latih: (452, 526) uji: (112, 526) kepadatan latih = 2.35%
kejadian di kedua split: 0


## 5. Pemeriksaan kebocoran
Pembagian per kejadian tidak mencegah kemiripan antar-templat. Periksa berapa templat uji yang juga ada di latih.

In [6]:
print("templat uji:", uji["template_id"].nunique(), "| ada juga di latih:", len(set(uji["template_id"]) & set(latih["template_id"])))
import pandas as pd
pd.crosstab(df["post_type"], df["label"])

templat uji: 61 | ada juga di latih: 59


label,Emergency,Non-Emergency
post_type,,
bantuan,74,0
berita,0,49
candaan,0,37
edukasi,0,43
info_lama,0,40
laporan,117,0
opini,0,43
peringatan,0,39
tidak_terkait,0,122


**Catatan keputusan (contoh):** (1) buang duplikat persis; (2) pembagian per `event_id`; (3) jangan pakai `post_type`
atau `template_id` sebagai fitur (diturunkan dari label/pembangkit data); (4) laporkan recall Emergency, bukan hanya akurasi.

## Latihan
Ulangi sel 3 pada `muat_uji_tangan()` dengan ambang DF yang lebih kecil. Mengapa katanya berbeda dari data sintetis?